# Wykorzystanie rozkładów macierzy do rozwiązywania oznaczonych układów równań

Poznane na wykładzie rozkłady (faktoryzacje) macierzy możemy wykorzystać do rozwiązywania układów równań liniowych. Dzięki wykorzystaniu specjalnych własności macierzy występujących w rozkładzie jesteśmy w stanie w łatwiejszy sposób odwrócić macierz i tym samym zminimalizować błąd.


**Zadanie 1.**

Rozważmy układ równań $Ax=b$, w którym:
* $A$ jest macierzą Hilberta o wymiarach 15x15.
* $A$ jest macierzą wartości losowych z przedziału $[0,100]$ o wymiarach 100x100,  1000x1000 i 1000000x1000000.
* $b$ jest wektorem wartości losowych, odpowiednio, o wymiarach 15x1, 100x1, 1000x1 i 1000000x1.

1. Oblicz współczynnik uwarunkowania macierzy $A$ i oceń jej uwarunkowanie.
2. Rozwiąż układ równań następującymi metodami:
    * z użyciem jawnej odwrotności $A$.
    * korzystając z rozkładu [LU](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.lu.html) (uwaga na macierz permutacji!):
        * z wykorzystaniem odwrotności L i U.
        * z użyciem jedynie odwrotności i metody podstawiania wstecznego.
    * korzystając z rozkładu [QR](https://numpy.org/doc/stable/reference/generated/numpy.linalg.qr.html):
        * z wykorzystaniem odwrotności Q i R.
        * z użyciem jedynie odwrotności Q i metody podstawiania wstecznego.
    * za pomocą jednej z metod iteracyjnych z poprzedniego ćwiczenia.
    * za pomocą metody [`solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html) z pakietu NumPy.
3. Porównaj otrzymane wyniki. W tym celu oblicz normy z residuuów otrzymanych dla każdego z rozwiązań. Którą z metod cechuje najwyższa dokladność?
4. Przeprowadź porównanie wydajności ww. metod. Zmierz czas wykonania każdej metody. Aby otrzymać bardziej sensowny wynik należy powtórzyć obliczenia w pętli (np. 100 lub 1000 razy) i uśrednić wynik. Do pomiaru czasu wykonania możesz wykorzystać pakiet `time`.
Wskazówka: Do rozwiązania układu z macierzą trójkątną możesz wykorzystać funkcję [`scipy.linalg.solve_triangular`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_triangular.html).

        
        


In [ ]:
%matplotlib inline
import numpy as np
import scipy.linalg as la
import time

def solve_and_benchmark(A, b, n_repeats=10, label=""):
    results = {}

    # 1. Jawna odwrotność
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        x_inv = np.linalg.inv(A) @ b
    results['Jawna odwrotność'] = (np.linalg.norm(A @ x_inv - b), (time.perf_counter()-t0)/n_repeats)

    # 2a. LU — odwrotności L i U
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        P, L, U = la.lu(A)
        Pb = P.T @ b
        x_lu_inv = np.linalg.inv(U) @ np.linalg.inv(L) @ Pb
    results['LU (odwrotności L,U)'] = (np.linalg.norm(A @ x_lu_inv - b), (time.perf_counter()-t0)/n_repeats)

    # 2b. LU — podstawianie wsteczne
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        P, L, U = la.lu(A)
        Pb = P.T @ b
        y  = la.solve_triangular(L, Pb, lower=True)
        x_lu_tri = la.solve_triangular(U, y)
    results['LU (podstawianie)'] = (np.linalg.norm(A @ x_lu_tri - b), (time.perf_counter()-t0)/n_repeats)

    # 3a. QR — odwrotności Q i R
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        Q, R = np.linalg.qr(A)
        x_qr_inv = np.linalg.inv(R) @ np.linalg.inv(Q) @ b
    results['QR (odwrotności Q,R)'] = (np.linalg.norm(A @ x_qr_inv - b), (time.perf_counter()-t0)/n_repeats)

    # 3b. QR — Q^T + podstawianie (Q ortogonalna => Q^{-1} = Q^T)
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        Q, R = np.linalg.qr(A)
        x_qr_tri = la.solve_triangular(R, Q.T @ b)
    results['QR (Q^T + podstawianie)'] = (np.linalg.norm(A @ x_qr_tri - b), (time.perf_counter()-t0)/n_repeats)

    # 4. numpy.linalg.solve
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        x_solve = np.linalg.solve(A, b)
    results['numpy.solve'] = (np.linalg.norm(A @ x_solve - b), (time.perf_counter()-t0)/n_repeats)

    print(f"\n{'='*65}")
    print(f"Macierz {label} ({A.shape[0]}x{A.shape[1]}),  cond(A) = {np.linalg.cond(A):.2e}")
    print(f"{'='*65}")
    print(f"{'Metoda':<30} {'||r||':>15} {'Czas [ms]':>12}")
    print(f"{'-'*65}")
    for method, (res, t) in results.items():
        print(f"{method:<30} {res:>15.2e} {t*1000:>12.4f}")

# Macierz Hilberta 15x15
np.random.seed(42)
n = 15
A_hilb = la.hilbert(n)
b_hilb = np.random.rand(n)
solve_and_benchmark(A_hilb, b_hilb, n_repeats=100, label="Hilberta")

# Macierze losowe
for size, reps in [(100, 100), (1000, 10)]:
    A_rand = np.random.uniform(0, 100, (size, size))
    b_rand = np.random.rand(size)
    solve_and_benchmark(A_rand, b_rand, n_repeats=reps, label="losowa")

print("\nUwaga: Macierz 1000000x1000000 wymaga ~8 TB RAM — pominięto.")

# Interpolacja

**Zadanie 2.**

Przeprowadź interpolacje poniższych funkcji $f(x)$ za pomocą wielomianów interpolacyjnych Lagrange'a ([`scipy.interpolate.lagrange`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.lagrange.html)). Stwórz wykresy funkcji interpolacyjnych i zaznacz na nich punkty, w ktorych dokonano oceny wartości funkcji.

Pierwsza funkcja:

* $f(0) = 1$,
* $f(0.25) = 1.64872$,
* $f(0.5) = 2.71828$,
* $f(0.75) = 4.48169.$

Oblicz $f(0.43)$.

Druga funkcja:

* $f_2(0.1) = 0.62049958$,
* $f_2(0.2) = -0.28398668$,
* $f_2(0.3) = 0.00660095$,
* $f_2(0.4) = 0.24842440$. 

Oblicz $f_2(0.25)$.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import lagrange

# --- Funkcja 1 ---
x1 = np.array([0.0,  0.25,     0.5,     0.75])
y1 = np.array([1.0,  1.64872,  2.71828, 4.48169])
p1 = lagrange(x1, y1)

val1 = float(p1(0.43))

x_plot = np.linspace(-0.05, 0.80, 400)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(x_plot, p1(x_plot), 'b-', lw=2, label="Wielomian Lagrange'a")
plt.scatter(x1, y1, color='red', zorder=5, s=80, label='Węzły interpolacji')
plt.scatter(0.43, val1, color='green', zorder=6, marker='*', s=200,
            label=f'f(0.43) ≈ {val1:.5f}')
plt.title("Funkcja 1 — Interpolacja Lagrange'a")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.legend(); plt.grid(True)

print(f"Funkcja 1:  f(0.43) ≈ {val1:.5f}  (e^0.43 = {np.exp(0.43):.5f})")

# --- Funkcja 2 ---
x2 = np.array([0.1,          0.2,         0.3,        0.4])
y2 = np.array([0.62049958,  -0.28398668,  0.00660095, 0.24842440])
p2 = lagrange(x2, y2)

val2 = float(p2(0.25))

x_plot2 = np.linspace(0.05, 0.45, 400)
plt.subplot(1, 2, 2)
plt.plot(x_plot2, p2(x_plot2), 'b-', lw=2, label="Wielomian Lagrange'a")
plt.scatter(x2, y2, color='red', zorder=5, s=80, label='Węzły interpolacji')
plt.scatter(0.25, val2, color='green', zorder=6, marker='*', s=200,
            label=f'f₂(0.25) ≈ {val2:.5f}')
plt.title("Funkcja 2 — Interpolacja Lagrange'a")
plt.xlabel("x"); plt.ylabel("f₂(x)"); plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Funkcja 2:  f₂(0.25) ≈ {val2:.5f}")

***Zadanie 3.***

Rozważmy funkcję $f(x)=\frac{1}{25x^2+1}$. 

Przeprowadź interpolacje funkcji $f$ w przedziale $[-2,2]$ wielomianem Lagrange'a oraz funkcjami sklejanymi 3 stopnia w:
- 21 równoodległych węzłach,
- 21 węzłach [Czebyszewa](https://pl.wikipedia.org/wiki/Węzły_Czebyszewa).

**Wskazówka** Interpolację funkcjami sklejanymi możesz przeprowadzić za pomocą funkcji [`interp1d`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.interp1d.html#scipy.interpolate.interp1d).

Umieść wielomian interpolacyjny, oryginalną funkcję $f$ oraz węzly interpolacyjne na wspólnym wykresie (jeden wykres dla metody Lagrange'a oraz jeden dla funkcji sklejanych). Porównaj otrzymane rezultaty. Przeprowadź te same działania dla przedziału $x\in[-5,5]$. Jakie problemy możesz zauważyć na otrzymanych wykresach?

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import lagrange, interp1d

def chebyshev_nodes(a, b, n):
    k = np.arange(1, n + 1)
    return 0.5*(a + b) + 0.5*(b - a)*np.cos((2*k - 1)*np.pi / (2*n))

f = lambda x: 1.0 / (25*x**2 + 1)
n_nodes = 21

for a, b in [(-2, 2), (-5, 5)]:
    x_dense = np.linspace(a, b, 1000)
    f_dense = f(x_dense)

    x_eq   = np.linspace(a, b, n_nodes)
    x_cheb = chebyshev_nodes(a, b, n_nodes)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Interpolacja f(x) = 1/(25x²+1)  na  [{a}, {b}]", fontsize=13)

    for col, (x_nodes, node_label) in enumerate([
            (x_eq,   "równoodległe"),
            (x_cheb, "Czebyszewa")]):

        y_nodes = f(x_nodes)

        # Lagrange
        p_lag = lagrange(x_nodes, y_nodes)
        y_lag = np.clip(p_lag(x_dense), -2, 2)

        axes[0, col].plot(x_dense, f_dense, 'k-',  lw=2,   label='f(x)')
        axes[0, col].plot(x_dense, y_lag,   'b--', lw=1.5, label='Lagrange')
        axes[0, col].scatter(x_nodes, y_nodes, color='red', zorder=5, s=25, label='Węzły')
        axes[0, col].set_ylim(-1.5, 1.5)
        axes[0, col].set_title(f"Lagrange — węzły {node_label}")
        axes[0, col].legend(fontsize=8); axes[0, col].grid(True)

        # Funkcje sklejane 3. stopnia
        # bounds_error=False + fill_value='extrapolate' obsługuje węzły nie obejmujące brzegów
        spl   = interp1d(x_nodes, y_nodes, kind='cubic',
                         bounds_error=False, fill_value='extrapolate')
        y_spl = spl(x_dense)

        axes[1, col].plot(x_dense, f_dense, 'k-',  lw=2,   label='f(x)')
        axes[1, col].plot(x_dense, y_spl,   'g--', lw=1.5, label='Sklejane 3°')
        axes[1, col].scatter(x_nodes, y_nodes, color='red', zorder=5, s=25, label='Węzły')
        axes[1, col].set_ylim(-1.5, 1.5)
        axes[1, col].set_title(f"Sklejane 3° — węzły {node_label}")
        axes[1, col].legend(fontsize=8); axes[1, col].grid(True)

    plt.tight_layout()
    plt.show()

print("Wnioski:")
print("• Węzły równoodległe + Lagrange: silne oscylacje przy brzegach (zjawisko Rungego),")
print("  szczególnie widoczne dla przedziału [-5, 5].")
print("• Węzły Czebyszewa + Lagrange: oscylacje znacznie zredukowane.")
print("• Funkcje sklejane: stabilne dla obu rozmieszeń węzłów — brak zjawiska Rungego.")

***Zadanie 4.***

Kierowca jadący z miasta A do miasta B, zauważywszy na drodze fotoradar, zaczął gwałtownie hamować. Przebieg jego położenia, zarejestrowany przez nawigację, pokazano w poniższej tabeli. Wiedząc, że radar znajduje się w punkcie o współrzędnej 79.6 m, oszacuj kiedy kierowca minął fotoradar (w tym celu skorzystaj z jednej z metod z laboratorium 3) oraz z jaką prędkością wtedy jechał (wykorzystaj relację drogi i prędkości znaną z fizyki). 

|czas \[s\]|położenie \[m\]|
|--|--|
|0.0|0.0|
|1.0|42.7|
|2.0|73.2|
|3.0|92.5|

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import lagrange
from scipy.optimize import brentq

# Dane z tabeli
t_data = np.array([0.0,  1.0,  2.0,  3.0])
x_data = np.array([0.0, 42.7, 73.2, 92.5])

# Wielomian interpolacyjny Lagrange'a
p  = lagrange(t_data, x_data)
dp = p.deriv()   # pochodna = prędkość

# Czas minięcia fotoradaru: p(t) = 79.6
t_radar = brentq(lambda t: p(t) - 79.6, 1.0, 3.0)
v_radar = float(dp(t_radar))

print(f"Czas minięcia fotoradaru:   t ≈ {t_radar:.4f} s")
print(f"Prędkość przy fotoradarze:  v ≈ {v_radar:.2f} m/s  =  {v_radar*3.6:.2f} km/h")

# Wykresy
t_plot = np.linspace(0, 3, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(t_plot, p(t_plot), 'b-', lw=2, label="Interpolacja Lagrange'a")
axes[0].scatter(t_data, x_data, color='red', zorder=5, s=80, label='Dane pomiarowe')
axes[0].axhline(79.6, color='orange', linestyle='--', label='Fotoradar (79.6 m)')
axes[0].axvline(t_radar, color='green', linestyle=':', label=f't ≈ {t_radar:.3f} s')
axes[0].set_xlabel("Czas [s]"); axes[0].set_ylabel("Położenie [m]")
axes[0].set_title("Położenie kierowcy"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(t_plot, dp(t_plot), 'r-', lw=2, label="v(t) = x'(t)")
axes[1].axvline(t_radar, color='green', linestyle=':', label=f't ≈ {t_radar:.3f} s')
axes[1].scatter([t_radar], [v_radar], color='green', zorder=6, s=120,
                label=f'v ≈ {v_radar:.2f} m/s\n({v_radar*3.6:.1f} km/h)')
axes[1].set_xlabel("Czas [s]"); axes[1].set_ylabel("Prędkość [m/s]")
axes[1].set_title("Prędkość kierowcy"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.show()

**Zadanie dla zainteresowanych programowaniem funkcyjnym w Pythonie**

Stwórz funkcję znajdującą wielomian interpolacyjny metodą Lagrange'a. Funkcja powinna przyjmować dwie listy:
* listę argumentów ($x$-ów)
* listę wartości ($y$-ów).  


Po wykonaniu obliczeń funkcja powinna zwracać wielomian w postaci obiektu typu **funkcja** (a nie `numpy.Polynomial`).

**Wskazówka** Wykorzystaj wyrażenia *lambda*. Dla ułatwienia możesz się też posłużyć pakietami `operator` i `functools`.


In [ ]:
import numpy as np
from functools import reduce
import operator

def lagrange_interpolation(xs, ys):
    """
    Zwraca wielomian interpolacyjny Lagrange'a jako obiekt funkcyjny (lambda).
    xs - lista/tablica argumentów węzłowych
    ys - lista/tablica wartości węzłowych
    """
    xs = list(xs)
    ys = list(ys)
    n  = len(xs)

    def basis(i):
        """i-ty wielomian bazowy L_i(x)"""
        num_terms   = [(xs[i] - xs[j]) for j in range(n) if j != i]
        denominator = reduce(operator.mul, num_terms, 1)
        return lambda x: (
            reduce(operator.mul, [(x - xs[j]) for j in range(n) if j != i], 1)
            / denominator
        )

    bases = [basis(i) for i in range(n)]
    return lambda x: sum(ys[i] * bases[i](x) for i in range(n))


# --- Test na danych z Zadania 2 ---
xs1 = [0.0,  0.25,    0.5,     0.75]
ys1 = [1.0,  1.64872, 2.71828, 4.48169]

p_custom = lagrange_interpolation(xs1, ys1)

print("Test funkcji lagrange_interpolation:")
print(f"  f(0.43)  = {p_custom(0.43):.5f}  (e^0.43 = {np.exp(0.43):.5f})")
print(f"  f(0.00)  = {p_custom(0.0):.5f}   (oczekiwane: 1.00000)")
print(f"  f(0.50)  = {p_custom(0.5):.5f}   (oczekiwane: 2.71828)")

# Porównanie z scipy.interpolate.lagrange
from scipy.interpolate import lagrange
p_scipy = lagrange(xs1, ys1)
print(f"\nPorównanie z scipy.interpolate.lagrange:")
print(f"  Własna impl.: f(0.43) = {p_custom(0.43):.8f}")
print(f"  scipy:        f(0.43) = {float(p_scipy(0.43)):.8f}")